# COMM061 NLP Group Coursework — Aiyoun Khan Anady
## Classical Models: TF-IDF + Logistic Regression, SVM, XGBoost

In [60]:
# From documentation of Hugging Face, the "dataset" module is needed.  
!pip install datasets

# Importing Libraries

In [61]:
from datasets import load_dataset 
import pandas as pd

dataset = load_dataset("surrey-nlp/BESSTIE-CW-26") # Importing the dataset as dataframe

train_df = pd.DataFrame(dataset['train'])
validation_df = pd.DataFrame(dataset['validation'])
test_df = pd.DataFrame(dataset['test'])

print(f"Train size: {len(train_df)}")
print(f"Validation size: {len(validation_df)}")
print(f"Test size: {len(test_df)}")

Train size: 3747
Validation size: 313
Test size: 2183


# 2. Data Preparation and Class Imbalance Check

## Justification 

As this is a dataset of reviews, there must be a class imbalance. Training on class imbalance can create biased resutls so first of all we need to see how bad the class imbalance is this is case. 

## Review

If we see the results, the distribution of the Sentiment classes are perfect to start with. Negative sentiment has 1907 counts and the other has 1840 sentiment which is nearly 50/50.

But the thing we need to worry about is the sarcasm data. The not sarcasm one has 3223 counts and the other has only 524 which is only 14% of the training examples are actually sarcastic. We will implement steps to fix that first.

In [62]:
print(f"Number of sentiment data: {train_df['Sentiment'].value_counts()}")
print(f"Number of sarcasm data: {train_df['Sarcasm'].value_counts()}")

Number of sentiment data: Sentiment
0.0    1907
1.0    1840
Name: count, dtype: int64
Number of sarcasm data: Sarcasm
0.0    3223
1.0     524
Name: count, dtype: int64


## 2.1 Classical Baselines

### Task 1 - Feature Extraction with TF-IDF

#### Feature Extraction - TF-IDF

We are using TF-IDF as a feature extraction process and we want the number to be 10,000. I have searched thorugh internet and found out that this is a good starting point but we will test with different feature numbers or simply said vocabulary list. 

#### Something to Note Down

We are using ".fit_transform()" for the training data to make the vocabualry list and ".transform()" to make the others. Because the ".transform()" converts the data using already learned vocabulary. So if we used the other one, that would have created a different vocabulary for the test data on which the model does not know anything.

In [63]:
from sklearn.feature_extraction.text import TfidfVectorizer

# To fed into the classifier, we need to separate the text and labels
X_train = train_df['text']
Y_train_sentiment = train_df['Sentiment']
Y_train_sarcasm = train_df['Sarcasm']

X_validation = validation_df['text']
Y_validation_sentiment = validation_df['Sentiment']
Y_validation_sarcasm = validation_df['Sarcasm']

X_test = test_df['text']
X_test_sentiment = test_df['Sentiment']
X_test_sarcasm = test_df['Sarcasm']

# Building TF-IDF features 
vectorizer = TfidfVectorizer(max_features=10000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_validation_tfidf = vectorizer.transform(X_validation)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Training matrix shape: {X_train_tfidf.shape}")
print(f"Validation matrix shape: {X_validation_tfidf.shape}")
print(f"Training matrix shape: {X_test_tfidf.shape}")

Training matrix shape: (3747, 10000)
Validation matrix shape: (313, 10000)
Training matrix shape: (2183, 10000)


### Task 2 - Logistic Regression - Sentiment

#### Result explanation

First of all we need to know that the *the sentiment class is balanced*. The result is expected to be good and, really they are. Lets analyse the results.

1. The precision actually came out good as a baseline model. When it predicted negative, nearly 85% was right and same goes for the positive one. Aound 84% was right when it predicted positive result. That is a good result to start with not going to lie.
2. As for the recall, the results were also good. Of all teh negative reviews, it found 85% of them and same goes for the positive reviews. Of all teh positive ones, it caught around 84% of them.
3. Moreover, great F1 scores for both the classes. 

In [64]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report 

# Trainning the model here
lr_model = LogisticRegression(max_iter=1000) # Lets start with 1000, we would comapre later
lr_model.fit(X_train_tfidf, Y_train_sentiment)

# Prediction on validation data 
Y_prediction = lr_model.predict(X_validation_tfidf)

# Evaluation of the classifier
print(f"Logistic regression on sentiment data")
print(classification_report(Y_validation_sentiment, Y_prediction))

Logistic regression on sentiment data
              precision    recall  f1-score   support

         0.0       0.85      0.85      0.85       160
         1.0       0.84      0.84      0.84       153

    accuracy                           0.85       313
   macro avg       0.85      0.85      0.85       313
weighted avg       0.85      0.85      0.85       313



### Task 3 - Logistic Regression - Sarcasm

#### Interpretation

If we see the result, you can see that we have got incredibel recall, precision, and f1-scores for finding the negative sarcasm only. And also we know the reason. It is for the *class imbalance* we have detected for the sarcasm distribution for the dataset. We have seen that nearly 85% of the data are not sarcastic and the model trained on that. This is the reason why the model could detect all the non-sarcasm examples but completely failed to detect the sarcastic one. This is one of the major reason we need to fix the class imbalance.  

In [65]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report 

# Trainning the model here
lr_model = LogisticRegression(max_iter=1000) # Lets start with 1000, we would comapre later
lr_model.fit(X_train_tfidf, Y_train_sarcasm)

# Prediction on validation data 
Y_prediction = lr_model.predict(X_validation_tfidf)

# Evaluation of the classifier
print(f"Logistic regression on sarcasm data")
print(classification_report(Y_validation_sarcasm, Y_prediction))

Logistic regression on sarcasm data
              precision    recall  f1-score   support

         0.0       0.86      1.00      0.92       269
         1.0       0.00      0.00      0.00        44

    accuracy                           0.86       313
   macro avg       0.43      0.50      0.46       313
weighted avg       0.74      0.86      0.79       313



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Task 4 - SVM - Sentiment

#### Interpretation 

The resutls are expected. Both the model has nearly identical results but the baseline resutls are good and nothing unexpected. 

In [66]:
from sklearn.svm import SVC

# Training SVM model here
svm_model = SVC(kernel="linear")
svm_model.fit(X_train_tfidf, Y_train_sentiment)

# Prediction on the validation data
Y_prediction = svm_model.predict(X_validation_tfidf)

# Evaluation of the classifier
print(f"SVM on sentiment data")
print(classification_report(Y_validation_sentiment, Y_prediction))

SVM on sentiment data
              precision    recall  f1-score   support

         0.0       0.84      0.85      0.85       160
         1.0       0.84      0.84      0.84       153

    accuracy                           0.84       313
   macro avg       0.84      0.84      0.84       313
weighted avg       0.84      0.84      0.84       313



### Task 4 - SVM - Sarcasm

#### Interpretation 

The result is expected again. The f1-score and the recall is better and we can say that the model is if fact trying a little bit. But there is one thing which is needed to be mentioned. The precision for the sarcastic reviews (67%) are legit better which is unexpected. Because the validation example might have contained 3-4 examples which the model guess right. That is why the precision is high but if we see the recall we can understand the main picture. It is extremely low as out of all actually sarcastic example, the model only guessed 5% of them. It is also because of the class imbalance problem.  

In [67]:
from sklearn.svm import SVC

# Training SVM model here
svm_model = SVC(kernel="linear")
svm_model.fit(X_train_tfidf, Y_train_sarcasm)

# Prediction on the validation data
Y_prediction = svm_model.predict(X_validation_tfidf)

# Evaluation of the classifier
print(f"SVM on sarcasm data")
print(classification_report(Y_validation_sarcasm, Y_prediction))

SVM on sarcasm data
              precision    recall  f1-score   support

         0.0       0.86      1.00      0.93       269
         1.0       0.67      0.05      0.09        44

    accuracy                           0.86       313
   macro avg       0.77      0.52      0.51       313
weighted avg       0.84      0.86      0.81       313



### Task 5 - RBF - Sentiment

#### Background

RBF basically takes advantage of higher dimensions to make a opimal margin to separate the data ponits which is better than SVM and Logistic Regression. 

#### Interpretation 

And the result is expected for the base model. As usual, the model did good. 

In [68]:
from sklearn.svm import SVC

# Training SVM model here
rbf_model = SVC(kernel="rbf")
rbf_model.fit(X_train_tfidf, Y_train_sentiment)

# Prediction on the validation data
Y_prediction = rbf_model.predict(X_validation_tfidf)

# Evaluation of the classifier
print(f"RBF on sentiment data")
print(classification_report(Y_validation_sentiment, Y_prediction))

RBF on sentiment data
              precision    recall  f1-score   support

         0.0       0.85      0.85      0.85       160
         1.0       0.84      0.84      0.84       153

    accuracy                           0.85       313
   macro avg       0.85      0.85      0.85       313
weighted avg       0.85      0.85      0.85       313



### Task 5 - RBF - Sarcasm

#### Interpretation 

Here we can see that, the RBF just gave up on the sarcasm data. And the problem we know that. The class imbalance is really creating a problem for us here. 

In [69]:
from sklearn.svm import SVC

# Training SVM model here
rbf_model = SVC(kernel="rbf")
rbf_model.fit(X_train_tfidf, Y_train_sarcasm)

# Prediction on the validation data
Y_prediction = rbf_model.predict(X_validation_tfidf)

# Evaluation of the classifier
print(f"RBF on sarcasm data")
print(classification_report(Y_validation_sarcasm, Y_prediction))

RBF on sarcasm data
              precision    recall  f1-score   support

         0.0       0.86      1.00      0.92       269
         1.0       0.00      0.00      0.00        44

    accuracy                           0.86       313
   macro avg       0.43      0.50      0.46       313
weighted avg       0.74      0.86      0.79       313



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Notes on performence (SVM vs RBF)

You can see that the SVM at least got 67% where the RBF just gave up on the sarcasm data. But as RBF supposed to perform better than the SVM, why did that happen?

When data is imbalanced like sarcasm (44 vs 269), the boundary naturally gets pushed towards the minority class (sarcastic) because SVM is trying to maximise the gap. This accidentally makes it slightly more willing to predict sarcastic on a few examples. Whereas, in the RBF, even if it takes the data to a new dimension, the class imbalance follows it there also. Moreover, it still is dominated by the majority class. 

### Task 6 - XGBoost Sentiment and Sarcasm

In [70]:
!pip install xgboost

#### Interpretation 

Worked on the XGBoost but found something which is worth mentioning. XGBoost really performed worse than the other models. This is out of expectations because this supposed to be better. But there are two major reason why it happened. Maybe, 

1. As XGBoost work on Gradient Boosted Decision Trees, each node is dependent on the previous node's error a lot. As there are a lot of nodes branching out on each other, small datasets or not enough data makes it learn from the training data which is a good example of overfitting. So in the validation, when the data slightly changes, it starts to crack and make wrong predictions.
2. It is because how the TF-IDF scores are so sparse which is often more than 99% of the zeros. It affect model which is based on tree like structures such as XGBoost. A binary decision tree splits a node by picking one feature f and a threshold t. Sending samples to the left if value[f] is less than t and to the right otherwise. For TF‑IDF, most documents has value[f] = 0.0 for most of the words. So the only threshold that can create any separation is if we set the threshold to somethinig like 0.3. If we set the left child to <= 0.3, then most of the zero sample will go to the left child. And the right child will get meaningful samples but extremely less in numbers. Why is this a problem? Here the left child has a lot of documents which is not pure and irrelevant. But in the right child, here are extremely minimal samples, but highly pure and relevant. To make complex patterns, the model has to make much more splits isolating each tiny groups of words. Now lets say we are talking about African Wildlife where we have two words which are "Elephants" and "Savannas" and here the word "Savannas" is a subword of "Elephants". If the threashold of the "Elephants" is > 0.3, then already in the first split the whole tree becomes highly imblanaced such as out of 1000 samples, 990 samples for the left and irrelevant child and the other 10 samples for the "Elephants" child. Also if you wanna consider for the subword, and the threshold for the subword "Savannas" is > 0.2, then the right child becomes more short and splitted. As the combination of "Elephants" and "Savannas" are short, the model easily remembers it and creates overfitting which crashes when seen new examples. 

#### Why Logistic Regression Avoid This?

Simply said, it does not need any tree like structure. It just learn thorugh the whole set of documents and adjusts its' weight parameters. Moreover, overfitting is reduced because rare conjunctions don't get their own dedicated leaf.

#### Tree-like Models and TF-IDF Scores Are A Bad Combination

Any tree like structures is not that good of a deal for features like TF-IDF scores. This is also a motivation of my part is to make it better by: 

1. Reducing dimentionality.
2. Using tree-friendly features like word counts instead of TF-IDF.
3. When the dataset is dense enough.

In [71]:
# Working on the sentiment data with XGBoost

from xgboost import XGBClassifier

# Training XGBoost model here
xgb_model = XGBClassifier(eval_metric='logloss')
xgb_model.fit(X_train_tfidf, Y_train_sentiment)

# Prediction on the validation data
Y_prediction = xgb_model.predict(X_validation_tfidf)

# Evaluation of the classifier
print(f"XGBoost on sentiment data")
print(classification_report(Y_validation_sentiment, Y_prediction))

XGBoost on sentiment data
              precision    recall  f1-score   support

         0.0       0.84      0.85      0.85       160
         1.0       0.84      0.84      0.84       153

    accuracy                           0.84       313
   macro avg       0.84      0.84      0.84       313
weighted avg       0.84      0.84      0.84       313



#### Interpretation 

We can see this is the best one for sarcasm dataset. Though it is a terrible score, but it did better than all of them. The main reason is the tree like structure. As each node word on the error of the previous node, like this, it slightly was able to improve that score. 

In [72]:
# Working on the sarcasm data with XGBoost

from xgboost import XGBClassifier

# Training XGBoost model here
xgb_model = XGBClassifier(eval_metric='logloss')
xgb_model.fit(X_train_tfidf, Y_train_sarcasm)

# Prediction on the validation data
Y_prediction = xgb_model.predict(X_validation_tfidf)

# Evaluation of the classifier
print(f"XGBoost on sarcasm data")
print(classification_report(Y_validation_sarcasm, Y_prediction))

XGBoost on sarcasm data
              precision    recall  f1-score   support

         0.0       0.88      0.97      0.92       269
         1.0       0.50      0.18      0.27        44

    accuracy                           0.86       313
   macro avg       0.69      0.58      0.59       313
weighted avg       0.83      0.86      0.83       313



## 2.2 Classical Baselines (Logistic Regression) - Class Imbalance Prevention using SMOTE

#### Background 

What is SMOTE? It is a way to artificially balance classes to make a proper class balance. How it does the thing? Lets say we have two examples which are example A: "Oh great, another Monday" and example B: "Oh wonderful, stuck in traffic again". What SMOTE does is it makes an artificial in between example which is "Oh great, stuck in traffic on Monday". It is not a real example, but it works you know. This way it makes plenty of examples and balances the classes. Even though it cannot change the raw texts, it does it mathematically. But it is good for a good example.

In [73]:
!pip install imbalanced-learn

### Task 1 - Fixing Sarcasm Class Imbalance using SMOTE

In [74]:
from imblearn.over_sampling import SMOTE

# Create the SMOTE object
smote = SMOTE(random_state=42) # 42 is the answer to life 

# Application of the SMOTE to only trainning data
X_train_smote, Y_train_sarcasm_smote = smote.fit_resample(X_train_tfidf, Y_train_sarcasm)

# The new distribution 
import numpy as np
unique, counts = np.unique(Y_train_sarcasm_smote, return_counts=True)
print(unique, counts)

[0. 1.] [3223 3223]


### Task 2 - Training Different Models Using Class Balance

1. First cell is for the Logistic Regression for both Sentiment and Sarcasm
2. Second cell is for the SVM for both Sentiment and Sarcasm
3. Third cell is for the RBF for both Sentiment and Sarcasm
4. Fourth cell is for the XGBoost for both Sentiment and Sarcasm

#### Interpretation 

If you look at the scores now, the f1 score gone from 0.00 to literally 43% just for a class imbalance change. Though this is not production level result but as for an improvement, a change like this is great. But there is a slight trade off. You can see that previously the f1 score was around 92% for the nor sarcastic data. But now it is 87%. A slight decrease because now we are focusing on minority class so it started to more generalise and this is the reason the majority class has taken a slight blow. But the trade off is worth it. 

In [75]:
'''
Logistic Regression - Sentiment Analysis
'''

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report 

# Trainning the model here
lr_model = LogisticRegression(max_iter=1000) # Lets start with 1000, we would comapre later
lr_model.fit(X_train_tfidf, Y_train_sentiment)

# Prediction on validation data 
Y_prediction = lr_model.predict(X_validation_tfidf)

# Evaluation of the classifier
print(f"{10 * "="} Logistic regression on sentiment data {10 * "="}")
print(classification_report(Y_validation_sentiment, Y_prediction))

'''
Logistic Regression With SMOTE - Sarcasm Analysis
'''

# Trainning the model here
lr_model = LogisticRegression(max_iter=1000) # Lets start with 1000, we would comapre later
lr_model.fit(X_train_smote, Y_train_sarcasm_smote)

# Prediction on validation data 
Y_prediction = lr_model.predict(X_validation_tfidf)

# Evaluation of the classifier
print(f"{10 * "="} Logistic regression with SMOTE on sarcasm data {10 * "="}")
print(classification_report(Y_validation_sarcasm, Y_prediction))

========== Logistic regression on sentiment data ==========
              precision    recall  f1-score   support

         0.0       0.85      0.85      0.85       160
         1.0       0.84      0.84      0.84       153

    accuracy                           0.85       313
   macro avg       0.85      0.85      0.85       313
weighted avg       0.85      0.85      0.85       313

========== Logistic regression with SMOTE on sarcasm data ==========
              precision    recall  f1-score   support

         0.0       0.92      0.83      0.87       269
         1.0       0.35      0.57      0.43        44

    accuracy                           0.79       313
   macro avg       0.63      0.70      0.65       313
weighted avg       0.84      0.79      0.81       313



## 2.3 Classical Baselines (Logistic Regression) - Class Imbalance Prevention using Class Weights

### Background 

What SMOTE does is it creates artificial examples. But with class weights, it does not create copies. Instead it introduces a per class penalty factor where the standard training minimizes the loos over all samples. What it does is if the model misclasify a minority example, it gives a larger penatly than misclassifying a majority one. It favors the minority class. So for example if the model misclassify a sarcastic example, that mistake will cost 6x more than a mistake in misclassifying a not sarcastic example. 

### Interpretation 

The result in the SMOTE and the Class Weights are nearly identical. No noticable changes to be cared about. One thing can be noticed which is the difference in the recall. In SMOTE it is 57% and in the Class Weights it is 64%. Not that of a increase but it is noticable. Why? 

In SMOTE the data is synthetic. It is not real. So when the model is being trained, there is some impurity in the learning. It learns a slightly blurred picture of what sarcasm looks like. Whereas, in the Class Weights, it learns from pure examples but just increasing the weights based on the proportion. But the training is done with pure data. This difference is projected in the recall scores. 

In [76]:
'''
Logistic Regression With Class Weights - Sarcasm Analysis
'''

# Trainning the model here
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_model.fit(X_train_tfidf, Y_train_sarcasm)

# Prediction on validation data 
Y_prediction = lr_model.predict(X_validation_tfidf)

# Evaluation of the classifier
print(f"{10 * "="} Logistic regression with Class Weights on sarcasm data {10 * "="}")
print(classification_report(Y_validation_sarcasm, Y_prediction))

========== Logistic regression with Class Weights on sarcasm data ==========
              precision    recall  f1-score   support

         0.0       0.93      0.79      0.85       269
         1.0       0.33      0.64      0.43        44

    accuracy                           0.77       313
   macro avg       0.63      0.71      0.64       313
weighted avg       0.85      0.77      0.79       313



## 2.3 Classical Baselines (XGBoost) - Class Imbalance Prevention using SMOTE and Class Weights in Sarcasm Data

In [77]:
'''
Working on the sarcasm data with XGBoost using SMOTE
'''

from xgboost import XGBClassifier

# Training XGBoost model here
xgb_model = XGBClassifier(eval_metric='logloss')
xgb_model.fit(X_train_smote, Y_train_sarcasm_smote)

# Prediction on the validation data
Y_prediction = xgb_model.predict(X_validation_tfidf)

# Evaluation of the classifier
print(f"XGBoost on sarcasm data using SMOTE")
print(classification_report(Y_validation_sarcasm, Y_prediction))

XGBoost on sarcasm data using SMOTE
              precision    recall  f1-score   support

         0.0       0.88      0.94      0.91       269
         1.0       0.33      0.18      0.24        44

    accuracy                           0.83       313
   macro avg       0.60      0.56      0.57       313
weighted avg       0.80      0.83      0.81       313



### Interpretation 

Same thing happened with the XGBoost. Its true that the F1 scores got hit a little bit compared to the Logistic Regression. And we discussed about it that tree like structure do not go with TF-IDF scores. Already it is in a disadvantage so from this there is a 4% decrease compared to the Logistic Regression models. 

In [78]:
'''
Working on the sarcasm data with XGBoost using Class Weights
'''

from xgboost import XGBClassifier

# Training XGBoost model here
xgb_model = XGBClassifier(eval_metric='logloss', scale_pos_weight=6.15)
xgb_model.fit(X_train_tfidf, Y_train_sarcasm)

# Prediction on the validation data
Y_prediction = xgb_model.predict(X_validation_tfidf)

# Evaluation of the classifier
print(f"XGBoost on sarcasm data using Class Weights")
print(classification_report(Y_validation_sarcasm, Y_prediction))

XGBoost on sarcasm data using Class Weights
              precision    recall  f1-score   support

         0.0       0.91      0.87      0.89       269
         1.0       0.38      0.48      0.42        44

    accuracy                           0.81       313
   macro avg       0.64      0.67      0.65       313
weighted avg       0.84      0.81      0.82       313



## Key Findings Till Now

1. Imbalance fixing always helps. Every model got improved with Imbalance Fixing.
2. Logistic Regression benifits more from imbalance fixing than XGBoost.
3. SMOTE vs Class Weight shows nearly identical results.
4. XGBoost performs good without any customization.

## 3 Middle Tier (GloVe + BiLSTM)

In [79]:
'''
BiLSTM needs a different set of library.
'''
!pip install torch torchtext

In [80]:
# Version checking because had an error

import torch
print(torch.__version__)

2.10.0+cu128


In [81]:
# Downloading the GloVe word vectors to use the context

import urllib.request 
import zipfile
import os

# Downloading the GloVe vectors (6B tokens, 100 dimensions)
urllib.request.urlretrieve("http://nlp.stanford.edu/data/glove.6B.zip", "glove.6B.zip")

# The file is in zip and we have to unzip that
with zipfile.ZipFile("glove.6B.zip", "r") as zip_ref:
    zip_ref.extractall("glove")

print(f"Done!!")

Done!!


In [82]:
# Check if GloVe is working as it supposed to be

import numpy as np 

# Loading the GloVe vectors
glove_embeddings  = {}

# Opening the GloVe in reading mode and with proper encoding
with open("glove/glove.6B.100d.txt", encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.array(values[1:], dtype="float32")
        glove_embeddings[word] = vector

print(f"Total words in GloVe: {len(glove_embeddings)}")
print(f"Vector dimensions: {glove_embeddings ["great"].shape}")

Total words in GloVe: 400000
Vector dimensions: (100,)


### Task 1 - Converting Words Into GloVe Vectors and Using It In BiLSTM 

Here, first of all we have a huge vocabulary list with proper context and it is GloVe. We have to convert the sentences into GloVe vectors. Basically we need to use the words provided in the list and the ones which are not in the list, we will assign a vector of zeros to that words. We call this out-of-vocabulary words which actually contribute nothing. Then we will fed each word of the whole sentence to the BiLSTM one at a time maintining the sequence. It reads the SEQUENCE of those vectors in both directions to understand context.

In [83]:
# Function to convert each sentence to the GloVe vector

def sentence_to_vector(sentence, embeddings, dim=100):
    words = sentence.lower().split()
    sequence = []
    for word in words:
        if word in embeddings:
            sequence.append(embeddings[word])
        else:
            sequence.append(np.zeros(dim))
    return np.array(sequence)

# Test in one sentence
sentence = "Great food was terrible"
sample = sentence_to_vector(sentence, glove_embeddings)
print(f"Sequence shape: {sample.shape}")
print(f"Number of words: {sample.shape[0]}")
print(f"Vector size per word: {sample.shape[1]}")

Sequence shape: (4, 100)
Number of words: 4
Vector size per word: 100


#### Problem 

One thing to consider that the PyTorch needs all the sentence to be in the same shape. Like for example (5, 1000) which means 5 words per sentence. But every review has different number of words. So we need padding. We need to chose how long each sentence would be and if the number of words < maximum number, we need ot pad this with zeros. As you see the results, the max length is 1035 words about which is good to forget. But the 95 percentile is good to go because it is only around 144 words and also it captures 95% of the sentence. So lets round this up into 150.

In [84]:
# Checking average sentence length in training data

lengths = []

for sentence in X_train:
    words = sentence.split() # Split into words
    word_count = len(words) # Count the words 
    lengths.append(word_count) # Append the counts 
    
print(f"Average length: {np.mean(lengths)}")
print(f"Max length: {np.max(lengths)}")
print(f"95th percentile: {np.percentile(lengths, 95)}")

Average length: 49.908193221243664
Max length: 1035
95th percentile: 143.69999999999982


#### Point to be Noted

Here we are using vstack because we want to pad the whole thing vertically which ensures order. Like this: 
[0.8, 0.2, 0.9, ...]  ← "great"
[0.5, 0.7, 0.2, ...]  ← "food"
[0.1, 0.2, 0.1, ...]  ← "was"
[0.0, 0.0, 0.0, ...]  ← padding
[0.0, 0.0, 0.0, ...]  ← padding
... until 150 rows total

In [85]:
MAX_LENGTH = 150 # Constant

def pad_sequence(sequence, max_length, dim=100):
    if len(sequence) >= max_length:
        return sequence[:max_length] # Cut the extra and keep until max length
    else:
        padding = np.zeros((max_length - len(sequence), dim))
        return np.vstack([sequence, padding]) # Pad if too short

In [86]:
# Function to convert all the splits into padded GloVe sequences 

def prepare_data(text_series):
    sequences = []
    for sentence in text_series:
        seq = sentence_to_vector(sentence, glove_embeddings)
        padded = pad_sequence(seq, MAX_LENGTH)
        sequences.append(padded)
    return np.array(sequences)

In [87]:
X_train_glove = prepare_data(X_train) # Preparing training data with GloVe
X_validation_glove = prepare_data(X_validation) # Preparing validation data wtih GloVe
X_test_glove = prepare_data(X_test) # Preparing test data with GloVe

print(f"Training shape: {X_train_glove.shape}")
print(f"Validation shape: {X_validation_glove.shape}")
print(f"Test shape: {X_test_glove.shape}")

Training shape: (3747, 150, 100)
Validation shape: (313, 150, 100)
Test shape: (2183, 150, 100)


### Task 2 - Building the BiLSTM Model

In [88]:
'''
Converting to tensors because we can take the advantage of the automatic gradient
descent of the PyTorch tensors and also the backpropagation. Also, for the GPU
support.
'''

import torch
import torch.nn as nn

# Setup device - use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Converting numpy arrays to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_glove).to(device)
X_validation_tensor = torch.FloatTensor(X_validation_glove).to(device)

# Converting labels to tensors
Y_train_sentiment_tensor = torch.FloatTensor(Y_train_sentiment.values).to(device)
Y_validation_sentiment_tensor = torch.FloatTensor(Y_validation_sentiment.values).to(device)
Y_train_sarcasm_tensor = torch.FloatTensor(Y_train_sarcasm.values).to(device)
Y_validation_sarcasm_tensor = torch.FloatTensor(Y_validation_sarcasm.values).to(device)

print(f"Input tensor shape: {X_train_tensor.shape}")
print(f"Label tensor shape: {Y_train_sentiment_tensor.shape}")

Using device: cuda
Input tensor shape: torch.Size([3747, 150, 100])
Label tensor shape: torch.Size([3747])


In [89]:
class BiLSTMClassifier(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim, num_layers, dropout):
        super(BiLSTMClassifier, self).__init__()

        self.lstm = nn.LSTM(input_size=input_dim,
                            hidden_size=hidden_dim,
                            num_layers=num_layers,
                            batch_first=True,
                            bidirectional=True,
                            dropout=dropout)

        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        lstm_out, (hidden, cell) = self.lstm(x)
        final = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        output = self.fc(final)
        return self.sigmoid(output)

# Creating model from the defined class
model = BiLSTMClassifier(input_dim=100,
                         hidden_dim=128,
                         output_dim=1,
                         num_layers=2,
                         dropout=0.3)

print(f"Model: {model}")

Model: BiLSTMClassifier(
  (lstm): LSTM(100, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (fc): Linear(in_features=256, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [90]:
# Training setup
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training function
def train_model(model, X, y, epochs=20, batch_size=32):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for i in range(0, len(X), batch_size):
            X_batch = X[i:i+batch_size]
            y_batch = y[i:i+batch_size]
            
            optimizer.zero_grad()
            predictions = model(X_batch).squeeze()
            loss = criterion(predictions, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        print(f"Epoch {epoch+1}/{epochs} Loss: {total_loss:.4f}")

In [91]:
'''
Training BiLSTM on Sentiment Data
'''
print("Training BiLSTM on Sentiment...")
sentiment_model = BiLSTMClassifier(input_dim=100,
                                   hidden_dim=128,
                                   output_dim=1,
                                   num_layers=2,
                                   dropout=0.3).to(device)

optimizer = torch.optim.Adam(sentiment_model.parameters(), lr=0.001)
train_model(sentiment_model, X_train_tensor, Y_train_sentiment_tensor)

Training BiLSTM on Sentiment...
Epoch 1/20 Loss: 64.1734
Epoch 2/20 Loss: 55.3927
Epoch 3/20 Loss: 52.7825
Epoch 4/20 Loss: 50.7830
Epoch 5/20 Loss: 49.4855
Epoch 6/20 Loss: 53.2250
Epoch 7/20 Loss: 46.3894
Epoch 8/20 Loss: 43.9831
Epoch 9/20 Loss: 42.4976
Epoch 10/20 Loss: 40.1588
Epoch 11/20 Loss: 40.0320
Epoch 12/20 Loss: 35.7173
Epoch 13/20 Loss: 35.7449
Epoch 14/20 Loss: 35.0245
Epoch 15/20 Loss: 32.7322
Epoch 16/20 Loss: 31.6641
Epoch 17/20 Loss: 28.5930
Epoch 18/20 Loss: 25.9757
Epoch 19/20 Loss: 24.4019
Epoch 20/20 Loss: 22.2737


In [92]:
def evaluate_model(model, X, y_true):
    model.eval()  # Switching to evaluation mode
    with torch.no_grad():  # No calculating gradients
        predictions = model(X).squeeze()
        predicted_labels = (predictions > 0.5).float()
    return predicted_labels.cpu().numpy()

# Evaluation on validation data (sentiment)
y_pred_sentiment = evaluate_model(sentiment_model, 
                                   X_validation_tensor, 
                                   Y_validation_sentiment_tensor)

print("BiLSTM - Sentiment (Validation)")
print(classification_report(Y_validation_sentiment, y_pred_sentiment))

BiLSTM - Sentiment (Validation)
              precision    recall  f1-score   support

         0.0       0.91      0.64      0.75       160
         1.0       0.71      0.93      0.81       153

    accuracy                           0.79       313
   macro avg       0.81      0.79      0.78       313
weighted avg       0.82      0.79      0.78       313



In [93]:
# Test labels for Sentiment Data
Y_test_sentiment = test_df['Sentiment']
Y_test_sarcasm = test_df['Sarcasm']

# Creating test tensor
X_test_tensor = torch.FloatTensor(X_test_glove).to(device)
Y_test_sentiment_tensor = torch.FloatTensor(Y_test_sentiment.values).to(device)

# Then evaluate on TEST data
Y_pred_sentiment_test = evaluate_model(sentiment_model,
                                       X_test_tensor,
                                       Y_test_sentiment_tensor)
print("BiLSTM - Sentiment (Test)")
print(classification_report(Y_test_sentiment, Y_pred_sentiment_test))

BiLSTM - Sentiment (Test)
              precision    recall  f1-score   support

         0.0       0.84      0.69      0.76      1117
         1.0       0.73      0.86      0.79      1066

    accuracy                           0.77      2183
   macro avg       0.78      0.78      0.77      2183
weighted avg       0.78      0.77      0.77      2183



In [99]:
'''
Training BiLSTM on Sarcasm Data
'''
sarcasm_model = BiLSTMClassifier(input_dim=100,
                                 hidden_dim=128,
                                 output_dim=1,
                                 num_layers=2,
                                 dropout=0.3).to(device)

# Handling class imbanalnce using POS Weight 
pos_weight = torch.tensor([3223/524]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(sarcasm_model.parameters(), lr=0.0001)

print("Training BiLSTM on Sarcasm")
train_model(sarcasm_model, X_train_tensor, Y_train_sarcasm_tensor, epochs=50)

# Evaluate BiLSTM on Sarcasm Data
y_pred_sarcasm = evaluate_model(sarcasm_model,
                                X_validation_tensor,
                                Y_validation_sarcasm_tensor)
print("BiLSTM - Sarcasm (Validation)")
print(classification_report(Y_validation_sarcasm, y_pred_sarcasm))

Training BiLSTM on Sarcasm
Epoch 1/50 Loss: 143.5426
Epoch 2/50 Loss: 139.3079
Epoch 3/50 Loss: 134.3239
Epoch 4/50 Loss: 132.8347
Epoch 5/50 Loss: 136.1309
Epoch 6/50 Loss: 131.9199
Epoch 7/50 Loss: 131.1719
Epoch 8/50 Loss: 132.7581
Epoch 9/50 Loss: 131.0990
Epoch 10/50 Loss: 131.9278
Epoch 11/50 Loss: 130.6950
Epoch 12/50 Loss: 131.2746
Epoch 13/50 Loss: 129.7771
Epoch 14/50 Loss: 130.7581
Epoch 15/50 Loss: 130.2270
Epoch 16/50 Loss: 131.1620
Epoch 17/50 Loss: 132.2339
Epoch 18/50 Loss: 131.1800
Epoch 19/50 Loss: 131.7340
Epoch 20/50 Loss: 131.9841
Epoch 21/50 Loss: 132.5268
Epoch 22/50 Loss: 130.9684
Epoch 23/50 Loss: 129.9073
Epoch 24/50 Loss: 130.0181
Epoch 25/50 Loss: 143.2218
Epoch 26/50 Loss: 135.2030
Epoch 27/50 Loss: 133.5235
Epoch 28/50 Loss: 132.4383
Epoch 29/50 Loss: 132.4281
Epoch 30/50 Loss: 131.4408
Epoch 31/50 Loss: 130.7457
Epoch 32/50 Loss: 131.4004
Epoch 33/50 Loss: 130.6166
Epoch 34/50 Loss: 130.1627
Epoch 35/50 Loss: 128.2031
Epoch 36/50 Loss: 128.6298
Epoch 37/5

In [116]:
Y_test_sarcasm_tensor = torch.FloatTensor(Y_test_sarcasm.values).to(device)
y_pred_bilstm_test = evaluate_model(sarcasm_model, X_test_tensor, Y_test_sarcasm_tensor)
print(classification_report(Y_test_sarcasm, y_pred_bilstm_test))

              precision    recall  f1-score   support

         0.0       0.94      0.73      0.82      1878
         1.0       0.30      0.71      0.42       305

    accuracy                           0.73      2183
   macro avg       0.62      0.72      0.62      2183
weighted avg       0.85      0.73      0.77      2183



### Intepretation of the Previous Results of GloVe + BiLSTM

After coding thorugh the whole thing, we can see some interesting results. First of all let us see a summary: 

SARCASM - TEST SET RESULTS:

Model                  | Test Macro F1 | Class 1 F1
-----------------------|---------------|------------
LR (class weights)     |     0.64      |    0.43  ← BEST
XGBoost (class weights)|     0.62      |    0.37
SVM (class weights)    |     0.61      |    0.35
BiLSTM (pos_weight)    |     0.62      |    0.42
LR (no fix)            |     0.46      |    0.00
SVM RBF (no fix)       |     0.46      |    0.00
XGBoost (no fix)       |     0.60      |    0.28
SVM linear (no fix)    |     0.51      |    0.09

It can be seen that the BiLSTM model is kinda sitting in the middle of all the tests we have done till now. That is interesting as it is expected that the model should score better than any of the models here. But why? There are some drawbacks which are:

1. The trainig examples. Models like needs loads of data to tune its parameters. But we had only 3747 data. It is too small for the model and it has extremely large parameters to learn. So, 3747 examples is nowhere near enough.
2. GloVe should have carried the whole process but there is a catch also. GloVe is basically trained on Wikipedia but our data is Reddit comments and Google Reviews. These are completely different. These kinda platform has slang, sarcasm, informal writing and GloVe barely understands these.
3. TF-IDF rewards RARE words such as "arvo", "benstokes". These rare slang words are actually the strongest and signals for variety detection!

## Needed to Add Later Section (All the Remaining Test Results)

In [100]:
from sklearn.metrics import classification_report

print("=" * 60)
print("SENTIMENT - TEST SET RESULTS")
print("=" * 60)

# Logistic Regression
lr_sentiment = LogisticRegression(max_iter=1000)
lr_sentiment.fit(X_train_tfidf, Y_train_sentiment)
y_pred = lr_sentiment.predict(X_test_tfidf)
print("\n--- LR - Sentiment ---")
print(classification_report(Y_test_sentiment, y_pred))

# SVM Linear
svm_linear_sentiment = SVC(kernel='linear')
svm_linear_sentiment.fit(X_train_tfidf, Y_train_sentiment)
y_pred = svm_linear_sentiment.predict(X_test_tfidf)
print("\n--- SVM Linear - Sentiment ---")
print(classification_report(Y_test_sentiment, y_pred))

# SVM RBF
svm_rbf_sentiment = SVC(kernel='rbf')
svm_rbf_sentiment.fit(X_train_tfidf, Y_train_sentiment)
y_pred = svm_rbf_sentiment.predict(X_test_tfidf)
print("\n--- SVM RBF - Sentiment ---")
print(classification_report(Y_test_sentiment, y_pred))

# XGBoost
xgb_sentiment = XGBClassifier(eval_metric='logloss')
xgb_sentiment.fit(X_train_tfidf, Y_train_sentiment)
y_pred = xgb_sentiment.predict(X_test_tfidf)
print("\n--- XGBoost - Sentiment ---")
print(classification_report(Y_test_sentiment, y_pred))

print("=" * 60)
print("SARCASM - TEST SET RESULTS")
print("=" * 60)

# LR with class weights (best LR)
lr_sarcasm = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_sarcasm.fit(X_train_tfidf, Y_train_sarcasm)
y_pred = lr_sarcasm.predict(X_test_tfidf)
print("\n--- LR Class Weights - Sarcasm ---")
print(classification_report(Y_test_sarcasm, y_pred))

# SVM Linear with class weights
svm_sarcasm = SVC(kernel='linear', class_weight='balanced')
svm_sarcasm.fit(X_train_tfidf, Y_train_sarcasm)
y_pred = svm_sarcasm.predict(X_test_tfidf)
print("\n--- SVM Linear Class Weights - Sarcasm ---")
print(classification_report(Y_test_sarcasm, y_pred))

# XGBoost with scale_pos_weight
xgb_sarcasm = XGBClassifier(eval_metric='logloss', scale_pos_weight=6.15)
xgb_sarcasm.fit(X_train_tfidf, Y_train_sarcasm)
y_pred = xgb_sarcasm.predict(X_test_tfidf)
print("\n--- XGBoost Class Weights - Sarcasm ---")
print(classification_report(Y_test_sarcasm, y_pred))

SENTIMENT - TEST SET RESULTS

--- LR - Sentiment ---
              precision    recall  f1-score   support

         0.0       0.81      0.88      0.84      1117
         1.0       0.86      0.79      0.82      1066

    accuracy                           0.83      2183
   macro avg       0.83      0.83      0.83      2183
weighted avg       0.83      0.83      0.83      2183


--- SVM Linear - Sentiment ---
              precision    recall  f1-score   support

         0.0       0.82      0.87      0.84      1117
         1.0       0.85      0.80      0.83      1066

    accuracy                           0.84      2183
   macro avg       0.84      0.83      0.83      2183
weighted avg       0.84      0.84      0.83      2183


--- SVM RBF - Sentiment ---
              precision    recall  f1-score   support

         0.0       0.81      0.89      0.85      1117
         1.0       0.87      0.78      0.82      1066

    accuracy                           0.84      2183
   macro avg  

## 4 McNemar's Test 

To prove the authenticity of the result we are going through some tests and one of them is McNemar's test. It looks at where two models disagree. If p < 0.05 then the difference is statistically significant and if p > 0.05, then the difference could be random luck.

In [101]:
!pip install statsmodels

In [106]:
from statsmodels.stats.contingency_tables import mcnemar
import numpy as np 

def run_mcnemar(Y_true, Y_pred1, Y_pred2, model1_name, model2_name):
    # Finding where the models agree and disagree
    correct1 = (Y_pred1 == Y_true)
    correct2 = (Y_pred2 == Y_true)

    # Building the contingency table
    n00 = sum(~correct1 & ~correct2)  # both wrong
    n01 = sum(~correct1 & correct2)   # model2 right, model1 wrong
    n10 = sum(correct1 & ~correct2)   # model1 right, model2 wrong
    n11 = sum(correct1 & correct2)    # both right

    table = [[n11, n10], [n01, n00]]
    result = mcnemar(table, exact=False)

    print(f"\n{model1_name} vs {model2_name}")
    print(f"Model 1 right, Model 2 wrong: {n10}")
    print(f"Model 2 right, Model 1 wrong: {n01}")
    print(f"p-value: {result.pvalue:.4f}")
    if result.pvalue < 0.05:
        print(f"Difference is real")
    else:
        print(f"Could be luck")

### Interpretation 

We have got some pretty interesting findings. Lets look at the first model differences. The difference in just not by luck and its real. Moreover, model 2 got more right than the model 1 which is expected but the model 1 has better Macro F1 but why?

It is a whole lot procedure and a whole lot calculation going on. But the short answer is SVM got more individual predictions correct overall such as lets say 92 out of 100 right. But LR did better on the minority classes such as Sarcasm and this is why the Macro F1 was higher.

The second one was as expected. The XGBoost definitely will guess better than the LR. 

But the third one is much more interesting. The SVM got more correct then the XGBoost one. This is kinda hilarious. And almost identical. So, the statistical equivalence of SVM and XGBoost (p=0.92) suggests that classical model errors are driven by inherent linguistic ambiguity in the data rather than model limitations — a ceiling that only contextualised models like RoBERTa might overcome.

In [117]:
# Running the comparisons

# Getting predictions from all models on test set
Y_true = Y_test_sarcasm.values

# LR predictions
lr_pred = lr_sarcasm.predict(X_test_tfidf)

# SVM predictions  
svm_pred = svm_sarcasm.predict(X_test_tfidf)

# XGBoost predictions
xgb_pred = xgb_sarcasm.predict(X_test_tfidf)

# BiLSTM predictions
Y_pred_bilstm_test = evaluate_model(sarcasm_model,
                                     X_test_tensor,
                                     Y_test_sarcasm_tensor)
Y_pred_bilstm_test = Y_pred_bilstm_test.astype(float)

# Running the McNemar Tests
run_mcnemar(Y_true, lr_pred, svm_pred, "LR", "SVM Linear")
run_mcnemar(Y_true, lr_pred, xgb_pred, "LR", "XGBoost")
run_mcnemar(Y_true, svm_pred, xgb_pred, "SVM Linear", "XGBoost")
run_mcnemar(Y_true, lr_pred, Y_pred_bilstm_test, "LR", "BiLSTM")
run_mcnemar(Y_true, svm_pred, Y_pred_bilstm_test, "SVM", "BiLSTM")
run_mcnemar(Y_true, xgb_pred, Y_pred_bilstm_test, "XGBoost", "BiLSTM")


LR vs SVM Linear
Model 1 right, Model 2 wrong: 97
Model 2 right, Model 1 wrong: 161
p-value: 0.0001
Difference is real

LR vs XGBoost
Model 1 right, Model 2 wrong: 154
Model 2 right, Model 1 wrong: 215
p-value: 0.0018
Difference is real

SVM Linear vs XGBoost
Model 1 right, Model 2 wrong: 200
Model 2 right, Model 1 wrong: 197
p-value: 0.9200
Could be luck

LR vs BiLSTM
Model 1 right, Model 2 wrong: 217
Model 2 right, Model 1 wrong: 151
p-value: 0.0007
Difference is real

SVM vs BiLSTM
Model 1 right, Model 2 wrong: 327
Model 2 right, Model 1 wrong: 197
p-value: 0.0000
Difference is real

XGBoost vs BiLSTM
Model 1 right, Model 2 wrong: 315
Model 2 right, Model 1 wrong: 188
p-value: 0.0000
Difference is real


## 5 MCC (Matthews Correlation Coefficient)

### Background 

The problem with the F1 scores are they are okay in extreme cases but it can be overly optimistic because it does not give a pentalty for false negative as heavily when negatives are easy. This problem is solved by MCC where all the four cells of a confution matrix are used. Moreover, F1 is not symmetric. F1 for class "positive" is different from F1 for class "negative" unless the confusion matrix is symmetric.
So in imbalanced data, we have to choose which class is "positive". Usually we pick the minority class. That's fine, but it means F1 ignores the majority class performance entirely. But MCC is symmetric and swapping the labels gives the same output and it treats both classes equally. 

### Interpretation 

It can be seen in the scores that the LR performs better than the rest two. While LR achieves slight better MCC (0.33 vs 0.32) and McNemar confirms statistically significant differences at prediction level, the practical performance gap between LR and BiLSTM is negligible. Classical models remain competitive with neural approaches on this small dataset 

In [119]:
# MCC Scores for Sarcasm class - Test Set

from sklearn.metrics import matthews_corrcoef

lr_mcc = matthews_corrcoef(Y_test_sarcasm, lr_pred)
svm_mcc = matthews_corrcoef(Y_test_sarcasm, svm_pred)
xgb_mcc = matthews_corrcoef(Y_test_sarcasm, xgb_pred)
bilstm_mcc = matthews_corrcoef(Y_test_sarcasm, y_pred_bilstm_test)

print(f"\nLR (class weights):      MCC = {lr_mcc:.4f}")
print(f"SVM (class weights):     MCC = {svm_mcc:.4f}")
print(f"XGBoost (class weights): MCC = {xgb_mcc:.4f}")
print(f"BiLSTM (pos_weight): MCC = {bilstm_mcc:.4f}")


LR (class weights):      MCC = 0.3288
SVM (class weights):     MCC = 0.2277
XGBoost (class weights): MCC = 0.2494
BiLSTM (pos_weight): MCC = 0.3241


## 6 Latency Benchmarking

### Background 

This is one of the most practical part of the whole part. It can directly help to the deployment part. We need to know which model is fast enough to predict in a deployed system. If a model takes longer time to predict after an user clicks, then we have to trash the model or make improvements. This test will give us a clue on which model to use. 

Moreover, we are using different batch sized to replicate the traffic. For example batch size 1 replicated a real time API where only one user requests to the model. Batch size 8 replicates a small group of request and so on and so forth. 

In [121]:
import time 
import numpy as np

def benchmark_model(model_fn, X, model_name, batch_sizes=[1, 8, 32, 128]):
    print(f"Model name: {model_name}")
    for batch_size in batch_sizes: 
        X_batch = X[:batch_size]
        times = []
        for _ in range(10): # Run 10 times and then average 
            start = time.time()
            model_fn(X_batch)
            end = time.time()
            times.append((end - start) * 1000)  # Convert to ms
        avg_time = np.mean(times)
        print(f"Batch size {batch_size:3d}: {avg_time:.2f} ms")


### Interpretation 

After seeing the results it can be finalized that the LR is incredibly fast and consisitant also. Then the batch size increases, the model shows almost no difference. On the other hand, SVM actually scales the worst. At batch 128, it goes to 77.54ms slower than the LR which is aroudn 60x slower. XGBoost does well on batch 1 and also scales good. It takes around 1.85ms to process a batch of 128 which is not that much bad. Last but not least, the BiLTSM also does well. It takes around 4.82ms for 1 batch and 16.47ms for a batch of 128 considering this model is huge and has a lot of parameters. The use of GPU helps it to better scale than SVM. For now, for a small dataset, LR is overall the best choice for deployment.

In [122]:
# LR
benchmark_model(lambda X: lr_sarcasm.predict(X), 
                X_test_tfidf, "Logistic Regression")

# SVM
benchmark_model(lambda X: svm_sarcasm.predict(X), 
                X_test_tfidf, "SVM Linear")

# XGBoost
benchmark_model(lambda X: xgb_sarcasm.predict(X), 
                X_test_tfidf, "XGBoost")

# BiLSTM
def bilstm_predict(X_numpy):
    X_tensor = torch.FloatTensor(X_numpy).to(device)
    with torch.no_grad():
        preds = sarcasm_model(X_tensor).squeeze()
    return (preds > 0.5).cpu().numpy()

benchmark_model(bilstm_predict,
                X_test_glove, "BiLSTM")

Model name: Logistic Regression
Batch size   1: 0.26 ms
Batch size   8: 0.21 ms
Batch size  32: 0.29 ms
Batch size 128: 0.23 ms
Model name: SVM Linear
Batch size   1: 1.32 ms
Batch size   8: 5.95 ms
Batch size  32: 19.69 ms
Batch size 128: 77.54 ms
Model name: XGBoost
Batch size   1: 0.70 ms
Batch size   8: 0.74 ms
Batch size  32: 0.97 ms
Batch size 128: 1.85 ms
Model name: BiLSTM
Batch size   1: 4.82 ms
Batch size   8: 5.26 ms
Batch size  32: 7.64 ms
Batch size 128: 16.47 ms
